In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import transforms
from torchvision.transforms import functional as TF
from PIL import Image
import torch
from collections import Counter
import numpy as np
from torch.amp import GradScaler, autocast

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Data Loader

In [3]:
ds = load_dataset("Ar4ikov/celebA_spoof")

Resolving data files:   0%|          | 0/93 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/65 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/93 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/65 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/78 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/56 [00:00<?, ?it/s]

In [4]:
img_size = 224
random_erase_prob = 0.25  
batch_size = 32
persistent_workers = True
prefetch_factor = 4
num_workers = 4
pin_memory = True
min_lr = 1e-6
warmup_epochs = 3

In [10]:
def get_transforms(is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomCrop(img_size),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=random_erase_prob),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

def letterbox(img, size):
    img = img if isinstance(img, Image.Image) else Image.open(img).convert("RGB")
    img.thumbnail((size, size), Image.Resampling.LANCZOS)

    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    x = (size - img.width) // 2
    y = (size - img.height) // 2
    canvas.paste(img, (x, y))

    return canvas

class CelebASpoofDataset(Dataset):
    def __init__(self, data, is_train=True):
        self.data = data
        self.transform = get_transforms(is_train)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = self.data[idx]['Filepath']
        label = 1 if self.data[idx]['Class'] == 'live' else 0

        img = letterbox(img, img_size)
        arr_img = self.transform(img)

        return img, arr_img, label

In [7]:
def dataset_collate(batch):
    images  = []
    targets = []
    for _, arr_img, label in batch:
        images.append(arr_img)
        targets.append(label)
    images  = torch.from_numpy(np.array(images)).type(torch.FloatTensor)
    targets = torch.from_numpy(np.array(targets)).long()
    return images, targets

In [11]:
train_dataset = CelebASpoofDataset(ds['train'])
val_dataset = CelebASpoofDataset(ds['valid'], is_train = False)

In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=prefetch_factor,
    collate_fn=dataset_collate,
    # drop_last=True,
)

In [13]:
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size * 2,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=prefetch_factor,
    collate_fn=dataset_collate,
)

### Model

In [14]:
model_name = "vit_base_patch16_224"
pretrained = True
dropout = 0.1
num_classes = 2

In [15]:
class ViTFaceAntiSpoofing(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        embed_dim = self.vit.num_features if hasattr(self.vit, 'num_features') else 768
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features = self.vit(x)
        return self.classifier(features)

In [16]:
model = ViTFaceAntiSpoofing().to(device)

In [17]:
sum(p.numel() for p in model.parameters())

86194946

### Load Model

In [18]:
checkpoint = torch.load("model_final.pth", map_location=device, weights_only=True)

In [19]:
model = ViTFaceAntiSpoofing().to(device)
model.load_state_dict(checkpoint)

<All keys matched successfully>

### Train

In [20]:
class AverageMeter:
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [21]:
label_smoothing = 0.1
learning_rate = 3e-4
weight_decay = 0.05
num_epochs = 30
mixed_precision = True
num_repeat_per_batch = 1
gradient_accumulation_steps = 1
max_grad_norm = 1.0

In [22]:
criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

In [23]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay, betas=(0.9, 0.999))

In [24]:
num_training_steps = len(train_loader) * num_epochs
warmup_steps = int(num_training_steps * warmup_epochs / num_epochs)

In [25]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_training_steps - warmup_steps, eta_min=min_lr)

In [26]:
num_training_steps

393690

In [27]:
scaler = GradScaler("cuda", enabled=mixed_precision)

In [28]:
losses = AverageMeter()
accs = AverageMeter()

for epoch in range(num_epochs):
    model.train()
    train_loss = []

    print()
    print(f"Start Train - {epoch}")
    print()

    for batch_index, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        for _ in range(num_repeat_per_batch):
            with autocast(device_type=device.type, enabled=mixed_precision):
                outputs = model(images)
                loss = criterion(outputs, labels)    
                
            scaler.scale(loss).backward()    
            
            if (batch_index + 1) % gradient_accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step() 
                
        with torch.no_grad():
            preds = torch.argmax(outputs, dim=1)
            acc = (preds == labels).float().mean()
        
        losses.update(loss.item(), images.size(0))
        accs.update(acc.item(), images.size(0))    

        post = {'loss': f'{losses.avg:.4f}', 'acc': f'{accs.avg:.4f}', 'lr': f'{optimizer.param_groups[0]["lr"]:.6f}'}
        print(post)
        # break
    # break


Start Train - 0

{'loss': '0.4949', 'acc': '0.8125', 'lr': '0.000300'}
{'loss': '0.5736', 'acc': '0.7344', 'lr': '0.000300'}


KeyboardInterrupt: 

### Save Model

In [34]:
# torch.save(model.state_dict(), "model_final.pth")